In [13]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer,  StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
tqdm.pandas(desc="Processing Rows")
import os
import secrets
import string
import gc
import time
import numpy as np
import matplotlib.pyplot as plt
import random
from torch.utils.data import DataLoader
import ast
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    r2_score
)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
hf_token = "os.environ['HF_TOKEN']" 

### Load in Model

In [2]:
model_id = "/common/data/models/meta-llama--Llama-3.2-3B"
from transformers import AutoTokenizer, AutoModelForCausalLM

hf_token = "os.environ['HF_TOKEN']" 

tokenizer = AutoTokenizer.from_pretrained(model_id,token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 254/254 [00:00<00:00, 403.79it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


### Data Reading and formatting

In [3]:

df=pd.read_csv("World_Ecological_2015_Metadata.csv")

columns = [
    'geonameid', 'name', 'asciiname', 'alternatenames', 
    'latitude', 'longitude', 'feature_class', 'feature_code', 
    'country_code', 'cc2', 'admin1', 'admin2', 'admin3', 'admin4', 
    'population', 'elevation', 'dem', 'timezone', 'modification_date'
]

df_geo = pd.read_csv('allCountries.txt', sep='\t', names=columns, low_memory=False)
df_landforms = df_geo[df_geo['feature_class'] == 'T']
combined_dataset=pd.read_csv("Global_Named_Landforms_With_Labels.csv")
combined_dataset=combined_dataset[['name','alternatenames','feature_code','ELU_LF_Des']]


/localscratch/471231/ipykernel_2115886/1638846168.py:12: DtypeWarning: Columns (9,10,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_dataset=pd.read_csv("Global_Named_Landforms_With_Labels.csv")


In [4]:
exact_6_class_map = {
    # 1: Plains
    'PLN': 1, 'PLNS': 1, 'FLT': 1, 'PANS': 1,
    
    # 2: Hills and Low Tablelands
    'HLL': 2, 'HLLS': 2, 'MND': 2, 'KNOB': 2,
    
    # 3: High Tablelands
    'PT': 3, 'PLAT': 3, 'MESA': 3, 'BUTT': 3,
    
    # 4: Mountains
    'RDG': 4, 'RDGS': 4, 'SADL': 4, 'DUNE': 4,
    
    # 5: Widely Spaced Mountains
    'MT': 5, 'MTS': 5, 'PK': 5, 'PKS': 5, 'CONE': 5, 'VOLC': 5, 'NUP': 5, 'RK': 5,
    
    # 7: Depressions or Basins
    'BSN': 7, 'BSNS': 7, 'DPR': 7, 'SINK': 7, 'CRQS': 7, 'CRATER': 7
}

combined_dataset['macro_class_id'] = combined_dataset['feature_code'].map(exact_6_class_map)
combined_dataset = combined_dataset.dropna(subset=['macro_class_id'])
combined_dataset['macro_class_id'] = combined_dataset['macro_class_id'].astype(int)
combined_dataset

,name,alternatenames,feature_code,ELU_LF_Des,macro_class_id
0,Roc Meler,"Roc Mele,Roc Meler,Roc Mélé",PK,Mountains,5
1,Pic de les Abelletes,"Pic de la Font-Negre,Pic de la Font-Nègre,Pic de les Abelletes",PK,Mountains,5
2,Pic de Meners,NaN,PK,Mountains,5
4,Roc de Port Dret,NaN,PK,Mountains,5
6,Roc del Xeig,NaN,RK,Mountains,5
...,...,...,...,...,...
1834929,South Rock,"Danzhu,Danzhu Shi,Danzhushi,South Rock,Tan-chu Shih,dan zhu,dan zhu shi,单柱,单柱石",RK,NaN,5
1834942,Whale Rock (historical),NaN,RK,NaN,5
1834949,Echo Point - Three Sisters,NaN,MT,NaN,5
1834953,Ipil Hill,"Ipil Hill,Ipil Seamount",HLL,NaN,2


In [5]:
combined_dataset=combined_dataset[~combined_dataset['macro_class_id'].isna()]
combined_dataset=combined_dataset[['name','macro_class_id']]
combined_dataset.columns=['Bridges Full Name','Topographic']
combined_dataset = combined_dataset.groupby('Bridges Full Name')['Topographic'].apply(lambda x: list(set(x))).reset_index()
ascii_mask = combined_dataset['Bridges Full Name'].astype(str).str.contains(r'^[\x00-\x7F]+$')
combined_dataset = combined_dataset[ascii_mask].reset_index(drop=True)
combined_dataset

,Bridges Full Name,Topographic
0,'Adade Yus Mountain,[5]
1,'Aqabat Nikid,[5]
2,'Elb Chouf,[4]
3,'Elb ech Chaoufa,[4]
4,'S Airde Beinn,[2]
...,...,...
618599,wzgorze na Katarynkach,[2]
618600,wzgorze nad Tuchlinkiem,[2]
618601,yak Gawa,[5]
618602,ytre Midtberget,[2]


### Creating  new Activation Datasets

#### Question prompt

In [9]:
question_dataset=combined_dataset.copy()
question_dataset['Bridges Full Name']=question_dataset['Bridges Full Name'].apply(lambda x: f"What is the landform of {x}")
question_dataset.to_csv("question_sensitivity_dataset.csv",index=False)
question_dataset.head()


,Bridges Full Name,Topographic
0,What is the landform of 'Adade Yus Mountain,[5]
1,What is the landform of 'Aqabat Nikid,[5]
2,What is the landform of 'Elb Chouf,[4]
3,What is the landform of 'Elb ech Chaoufa,[4]
4,What is the landform of 'S Airde Beinn,[2]


#### Random tokens

In [11]:
def generate_secure_token(length=5):
    alphabet = string.ascii_letters 
    return ''.join(secrets.choice(alphabet) for _ in range(length))

In [14]:
rand_token_dataset=combined_dataset.copy()
rand_token_dataset['Bridges Full Name']=rand_token_dataset['Bridges Full Name'].apply(lambda x: f"{generate_secure_token(random.randint(3,6))} {x}")
rand_token_dataset.to_csv("rand_token_dataset_sensitivity_dataset.csv",index=False)
rand_token_dataset.head()


,Bridges Full Name,Topographic
0,zVOJQt 'Adade Yus Mountain,[5]
1,PDDNT 'Aqabat Nikid,[5]
2,qFQ 'Elb Chouf,[4]
3,YBeD 'Elb ech Chaoufa,[4]
4,GOIyPA 'S Airde Beinn,[2]


#### Semantic Contradiction

In [18]:
macro_class_names = {
    1: "Plains",
    2: "Hills and Low Tablelands",
    3: "High Tablelands",
    4: "Mountains",
    5: "Widely Spaced Mountains",
    7: "Depressions or Basins",
}

def get_wrong_macro_class_string(ground_truth_labels):
    ground_truth_labels = set(ground_truth_labels)
    possible_wrong = [
        k for k in macro_class_names
        if k not in ground_truth_labels
    ]
    wrong_label = random.choice(possible_wrong)
    return macro_class_names[wrong_label]

In [20]:
contradiction_dataset=combined_dataset.copy()
contradiction_dataset['Bridges Full Name']=contradiction_dataset.apply(lambda x: f"{x['Bridges Full Name']} is a  {get_wrong_macro_class_string(x['Topographic'])}", axis=1)
contradiction_dataset.to_csv("contradiction_dataset_dataset_sensitivity_dataset.csv",index=False)
contradiction_dataset.head()


,Bridges Full Name,Topographic
0,'Adade Yus Mountain is a Hills and Low Tablelands,[5]
1,'Aqabat Nikid is a Depressions or Basins,[5]
2,'Elb Chouf is a Hills and Low Tablelands,[4]
3,'Elb ech Chaoufa is a Hills and Low Tablelands,[4]
4,'S Airde Beinn is a Mountains,[2]
